# Stratified DFT Hessians

Open this notebook from GitHub and use a fresh GPU runtime. Its first cell uploads the cumulative bundle downloaded by stage 02. It starts with two GPU4PySCF cases to calibrate wall time.

In [ ]:
from google.colab import files

uploaded = files.upload()
assert len(uploaded) == 1, 'Upload exactly one stage-02 .tar.gz bundle.'

In [ ]:
from pathlib import Path
import tarfile

uploaded_name = next(iter(uploaded))
input_bundle = Path('/content') / uploaded_name
if not input_bundle.is_file():
    input_bundle.write_bytes(uploaded[uploaded_name])
OUTPUT_ROOT = Path('/content/oa_audit_outputs')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
output_root_resolved = OUTPUT_ROOT.resolve()
with tarfile.open(input_bundle, 'r:gz') as archive:
    for member in archive.getmembers():
        target = (OUTPUT_ROOT / member.name).resolve()
        assert target == output_root_resolved or output_root_resolved in target.parents
        assert member.isfile() or member.isdir(), f'Unsupported archive entry: {member.name}'
    archive.extractall(OUTPUT_ROOT)

REPOSITORY_URL = 'https://github.com/jiaxi98/OAReactDiff.git'
REPOSITORY_REF = 'agent/oa-failure-audit'
REPO = Path('/content/OAReactDiff')
SCREEN_ROOT = OUTPUT_ROOT / 'horm_screen_generation_8x8_r2_j2'
SUBSET = SCREEN_ROOT / 'dft_subset.csv'
assert SUBSET.is_file(), f'The uploaded stage-02 bundle is missing {SUBSET}'
if not (REPO / '.git').is_dir():
    !GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 --branch {REPOSITORY_REF} {REPOSITORY_URL} {REPO}
else:
    !git -C {REPO} fetch --depth 1 origin {REPOSITORY_REF}
    !git -C {REPO} checkout --detach FETCH_HEAD
!git -C {REPO} rev-parse HEAD
!cd {REPO} && bash experiments/oa_failure_audit/setup_colab_dft.sh

In [ ]:
MAMBA = '/usr/local/bin/micromamba'
ENV_PREFIX = '/content/micromamba/envs/oa-dft'
BENCHMARK_OUTPUT = SCREEN_ROOT / 'dft_benchmark_two'
!cd {REPO} && {MAMBA} run -p {ENV_PREFIX} python experiments/oa_failure_audit/run_dft_hessian.py \
    --subset {SUBSET} --output-dir {BENCHMARK_OUTPUT} --backend gpu4pyscf \
    --max-candidates 2 --resume

In [ ]:
import csv
with (BENCHMARK_OUTPUT / 'dft_results.csv').open() as handle:
    benchmark_rows = list(csv.DictReader(handle))
mean_seconds = sum(float(row['wall_seconds']) for row in benchmark_rows) / len(benchmark_rows)
print(f'Mean: {mean_seconds / 60:.1f} minutes/case')
print(f'Projected 96 cases: {mean_seconds * 96 / 3600:.1f} serial GPU-hours')

In [ ]:
RUN_FULL_SUBSET = False
DFT_OUTPUT = SCREEN_ROOT / 'dft_full'
if RUN_FULL_SUBSET:
    !cd {REPO} && {MAMBA} run -p {ENV_PREFIX} python experiments/oa_failure_audit/run_dft_hessian.py \
        --subset {SUBSET} --output-dir {DFT_OUTPUT} --backend gpu4pyscf --resume

## Gate IRC by DFT evidence

Only a stationary DFT index-1 point is immediately IRC-eligible. A nonstationary index-1 raw sample is routed through TS optimization and another Hessian first.

In [ ]:
if RUN_FULL_SUBSET:
    DFT_ENRICHED = SCREEN_ROOT / 'dft_subset_with_results.csv'
    !cd {REPO} && {MAMBA} run -p {ENV_PREFIX} python experiments/oa_failure_audit/merge_screening.py \
        --manifest {SUBSET} --screen dft={DFT_OUTPUT / 'dft_results.csv'} \
        --output {DFT_ENRICHED} --overwrite
    IRC_WORKLIST = SCREEN_ROOT / 'irc_worklist.csv'
    !cd {REPO} && {MAMBA} run -p {ENV_PREFIX} python experiments/oa_failure_audit/prepare_irc_worklist.py \
        --manifest {DFT_ENRICHED} --output {IRC_WORKLIST} --overwrite

## Download the DFT results to this computer

This cell packages all three stages, including completed DFT artifacts and any IRC worklist, into one cumulative browser download for local analysis.

In [ ]:
import hashlib
import shutil
from google.colab import files

bundle_path = Path(shutil.make_archive(
    '/content/oa_audit_03_dft_hessian_generation_8x8_r2_j2',
    'gztar',
    root_dir=OUTPUT_ROOT,
    base_dir='.',
))
digest = hashlib.sha256()
with bundle_path.open('rb') as handle:
    for chunk in iter(lambda: handle.read(1024 * 1024), b''):
        digest.update(chunk)
print(f'{bundle_path.name}: {bundle_path.stat().st_size / 1024**2:.2f} MiB')
print(f'sha256: {digest.hexdigest()}')
files.download(str(bundle_path))